# 5.3 · 多轮往返请求（MRTR）与征询

**Multi round-trip requests (MRTR) and elicitation**

> **学完本课你能：** 端到端地实现并驱动 MRTR 流程（InputRequiredResult、inputResponses、受保护的 requestState），并安全地使用表单模式和 URL 模式的征询，包括为破坏性工具设置确认关卡。

官方阅读材料：[Spec: Multi round-trip requests](https://modelcontextprotocol.io/specification/2026-07-28/basic/patterns/mrtr) · [Spec: Elicitation](https://modelcontextprotocol.io/specification/2026-07-28/client/elicitation)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 带确认关卡的破坏性工具（手工实现 MRTR）
我们显式地编写整个流程，让每个字段都清晰可见。`delete_repository` 在第一次调用时绝不会删除任何东西。它会：

- 检查客户端是否声明了表单模式征询（Form-mode Elicitation）（这是规范中的 MUST；手工编写的代码必须自己做这项检查），
- 返回一个 `InputRequiredResult`，要求用户<strong>输入仓库名称</strong>，
- 在重试时，只有在 `accept` 且名称匹配时才执行操作；输错时重新提问；对 `decline`/`cancel` 进行报告。

`ctx.request_state` 保存着我们上一轮生成的内容。SDK 会在传出时加密它，并在传回时校验它。

In [ ]:
import json
import logging

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import Context

logging.getLogger("mcp").setLevel(logging.CRITICAL)


def wire(model) -> str:
    return json.dumps(model.model_dump(by_alias=True, exclude_none=True, mode="json"), indent=2)


REPOS = {"acme/web", "acme/prod", "acme/docs"}
github = MCPServer("github-admin")


def confirm_request(repo: str, note: str = "") -> types.ElicitRequest:
    return types.ElicitRequest(params=types.ElicitRequestFormParams(
        message=f"{note}This permanently deletes {repo}. Type the full repository name to confirm.",
        requested_schema={"type": "object",
                          "properties": {"repo_name": {"type": "string", "title": "Repository name",
                                                       "minLength": 1, "maxLength": 100}},
                          "required": ["repo_name"]}))


@github.tool()
def delete_repository(repo: str, ctx: Context) -> str | types.InputRequiredResult:
    """Permanently delete a repository (asks the user to confirm first)."""
    if repo not in REPOS:
        return f"No such repository: {repo}"
    caps = ctx.client_capabilities
    if caps is None or caps.elicitation is None:
        return "Refused: deletion needs user confirmation, but this client cannot ask the user."

    answer = (ctx.input_responses or {}).get("confirm_delete")
    if answer is None:                                     # round 1, or the client sent nothing
        return types.InputRequiredResult(input_requests={"confirm_delete": confirm_request(repo)},
                                         request_state=json.dumps({"repo": repo, "round": 1}))
    state = json.loads(ctx.request_state)                  # already verified by the SDK
    if answer.action == "decline":
        return f"Not deleted: the user declined deleting {state['repo']}."
    if answer.action == "cancel":
        return "Not deleted: the confirmation was dismissed. Ask again later if needed."
    if (answer.content or {}).get("repo_name") != state["repo"]:
        return types.InputRequiredResult(              # spec: re-ask rather than error
            input_requests={"confirm_delete": confirm_request(repo, "The name did not match. ")},
            request_state=json.dumps({"repo": repo, "round": state["round"] + 1}))
    REPOS.discard(state["repo"])
    return f"Deleted {state['repo']} after {state['round']} confirmation round(s)."


print("tool registered; repos:", sorted(REPOS))

### 2. 自动循环：`Client(elicitation_callback=...)`
高层的 `call_tool` 看到 `input_required` 后，会把每个输入请求路由到我们的回调，并带着 `inputResponses` 和 `requestState` 重试，直到结果完成。这就是多轮往返请求（MRTR）的自动驱动。我们脚本化的“用户”会演示四种场景。

In [ ]:
answers = []


async def confirm_dialog(context, params):
    print(f"   [dialog from github-admin] {params.message}")
    action, typed = answers.pop(0)
    print(f"   [user] {action} {typed or ''}")
    return types.ElicitResult(action=action, content={"repo_name": typed} if typed else None)


scenarios = {"decline": [("decline", None)],
             "cancel": [("cancel", None)],
             "typo, then correct": [("accept", "acme/wbe"), ("accept", "acme/web")]}

async with Client(github, elicitation_callback=confirm_dialog) as client:
    for label, script in scenarios.items():
        answers[:] = script
        result = await client.call_tool("delete_repository", {"repo": "acme/web"})
        print(f"{label:>20} -> {result.content[0].text}\n")
print("repos now:", sorted(REPOS))

### 3. 手动驱动并阅读原始 JSON
`client.session.call_tool(..., allow_input_required=True)` 会返回中间结果，而不是自动解决它。如果要持久化一个待回答的问题（例如一个聊天在等一位去吃午饭的用户），之后再恢复，就是这样做的。然后我们自己构建 `inputResponses`，并带着<strong>原样的</strong> `requestState`（请求状态）重试。

In [ ]:
REPOS.add("acme/web")

async with Client(github, elicitation_callback=confirm_dialog) as client:
    first = await client.session.call_tool("delete_repository", {"repo": "acme/docs"}, allow_input_required=True)
    print("ROUND 1 result:\n", wire(first))
    saved_state = first.request_state                      # opaque: store it, never parse it
    key = next(iter(first.input_requests))

    responses = {key: types.ElicitResult(action="accept", content={"repo_name": "acme/docs"})}
    final = await client.session.call_tool("delete_repository", {"repo": "acme/docs"},
                                           input_responses=responses, request_state=saved_state,
                                           allow_input_required=True)
    print("\nROUND 2 result:\n", wire(final))

我们生成的 `requestState` 是 `{"repo": "acme/docs", "round": 1}`，但客户端看到的只有一个 `v1.…` 令牌。SDK 的 `RequestStateBoundary` 用 AES-256-GCM 把它连同过期时间、方法、工具名称、参数摘要以及作为受众（Audience）的服务器名称一起封装了起来。

### 4. 制造故障：篡改并重放 `requestState`
把客户端当作攻击者。我们 (a) 翻转令牌中的字符，(b) 在<em>另一个</em>仓库上重放一个有效的令牌，(c) 发送服务器从未生成过的状态。每一种都会在我们的处理函数运行之前被拒绝，而且返回相同的错误，让攻击者得不到任何信息。这就是状态句柄劫持（State Handle Hijacking）的防御。

In [ ]:
from mcp.shared.exceptions import MCPError

async with Client(github, elicitation_callback=confirm_dialog) as client:
    first = await client.session.call_tool("delete_repository", {"repo": "acme/web"}, allow_input_required=True)
    good = first.request_state
    yes = {"confirm_delete": types.ElicitResult(action="accept", content={"repo_name": "acme/prod"})}
    attacks = {"bit-flipped token": (good[:-6] + ("A" * 6 if not good.endswith("AAAAAA") else "B" * 6), "acme/web"),
               "replayed on another repo": (good, "acme/prod"),
               "forged plaintext state": (json.dumps({"repo": "acme/prod", "round": 1}), "acme/prod")}
    for label, (state, repo) in attacks.items():
        try:
            await client.session.call_tool("delete_repository", {"repo": repo}, input_responses=yes,
                                           request_state=state, allow_input_required=True)
            print(f"{label:>26}: ACCEPTED (bad!)")
        except MCPError as err:
            print(f"{label:>26}: {err.error.code} {err.error.message} {err.error.data}")
print("acme/prod still exists:", "acme/prod" in REPOS)

### 5. 线路上：通过 stdio 传输的原始 JSON-RPC
这一次，客户端一侧没有 SDK。注意观察 <strong>id</strong>（先是 1，然后是 2）、`resultType` 的取值，以及状态被逐字节回传。`_meta` 信封在<em>每个</em>请求上都声明了 `elicitation: {form: {}}`，这就是逐请求的能力协商（Capability Negotiation）。

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "confirm_server.py").write_text(textwrap.dedent('''
    import json
    import mcp_types as types
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context

    server = MCPServer("confirm", log_level="CRITICAL")

    @server.tool()
    def drop_table(table: str, ctx: Context) -> str | types.InputRequiredResult:
        """Drop a database table after confirmation."""
        answer = (ctx.input_responses or {}).get("ok")
        if answer is None:
            return types.InputRequiredResult(
                input_requests={"ok": types.ElicitRequest(params=types.ElicitRequestFormParams(
                    message=f"Drop table {table}?",
                    requested_schema={"type": "object", "properties": {"confirm": {"type": "boolean"}},
                                      "required": ["confirm"]}))},
                request_state=json.dumps({"table": table}))
        ok = answer.action == "accept" and answer.content.get("confirm")
        return f"dropped {table}" if ok else f"kept {table}"

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "confirm_server.py")],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {"elicitation": {"form": {}}},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}


def rpc(message):
    print(">>>", json.dumps(message)[:400])
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    print("<<<", json.dumps(reply, indent=1)[:1100], "\n")
    return reply


call = {"name": "drop_table", "arguments": {"table": "users"}, "_meta": META}
r1 = rpc({"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": call})
r2 = rpc({"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {
    **call, "inputResponses": {"ok": {"action": "accept", "content": {"confirm": True}}},
    "requestState": r1["result"]["requestState"]}})

在线路上同样可以制造故障：发送一个<strong>没有</strong>声明任何能力的请求。这个服务器是手工实现的，不做检查，所以它照样会提问。这违反了规范中的 MUST，这也是第 1 节中的工具要检查 `ctx.client_capabilities` 的原因。

In [ ]:
no_caps = {**META, "io.modelcontextprotocol/clientCapabilities": {}}
r3 = rpc({"jsonrpc": "2.0", "id": 3, "method": "tools/call",
          "params": {"name": "drop_table", "arguments": {"table": "orders"}, "_meta": no_caps}})
print("server sent an elicitation to a client without the capability:", "inputRequests" in r3.get("result", {}))
proc.stdin.close()
exit_code = proc.wait(timeout=10)

### 6. 无状态的横向扩展：实例 A 提问，实例 B 完成
由于状态随请求一起传递，重试可以落到<strong>另一个服务器实例</strong>上，前提是它们共享封装密钥（并且使用相同的名称，因为名称是令牌的受众）。使用默认的每进程临时密钥时，实例 C 会拒绝 A 的令牌。这就是在负载均衡器后面配置 `RequestStateSecurity(keys=[...])` 的运维原因，也是水平扩展（Horizontal Scaling）的关键。较短的 TTL 还能演示过期。

In [ ]:
import secrets
import time

from mcp.server.request_state import RequestStateSecurity

SHARED_KEY = secrets.token_hex(32)


def make_instance(label, security):
    srv = MCPServer("transfers", request_state_security=security)

    @srv.tool()
    def transfer(amount: int, ctx: Context) -> str | types.InputRequiredResult:
        """Transfer money after the user approves the amount."""
        answer = (ctx.input_responses or {}).get("approve")
        if answer is None:
            return types.InputRequiredResult(
                input_requests={"approve": types.ElicitRequest(params=types.ElicitRequestFormParams(
                    message=f"Approve transfer of {amount} EUR?",
                    requested_schema={"type": "object", "properties": {"approve": {"type": "boolean"}},
                                      "required": ["approve"]}))},
                request_state=json.dumps({"amount": amount}))
        return f"instance {label}: transferred {json.loads(ctx.request_state)['amount']} EUR"
    return srv


a = make_instance("A", RequestStateSecurity(keys=[SHARED_KEY], ttl=2))
b = make_instance("B", RequestStateSecurity(keys=[SHARED_KEY], ttl=2))
c = make_instance("C", None)                                          # default: ephemeral per-process key
approve = {"approve": types.ElicitResult(action="accept", content={"approve": True})}

async with Client(a, elicitation_callback=confirm_dialog) as ca, Client(b) as cb, Client(c) as cc:
    pending = await ca.session.call_tool("transfer", {"amount": 50}, allow_input_required=True)
    done = await cb.session.call_tool("transfer", {"amount": 50}, input_responses=approve,
                                      request_state=pending.request_state, allow_input_required=True)
    print("A asked, B finished ->", done.content[0].text)
    try:
        await cc.session.call_tool("transfer", {"amount": 50}, input_responses=approve,
                                   request_state=pending.request_state, allow_input_required=True)
    except MCPError as err:
        print("A asked, C (own ephemeral key) ->", err.error.message)
    time.sleep(2.2)
    try:
        await cb.session.call_tool("transfer", {"amount": 50}, input_responses=approve,
                                   request_state=pending.request_state, allow_input_required=True)
    except MCPError as err:
        print("after TTL, B ->", err.error.message)

同一个有效令牌在其 TTL <em>之内</em>仍然可以在 B 上被重放。真正的转账还需要在服务器端记录“已使用”，因为规范指出，协议层面的措施并不能保证一次性使用。

### 7. URL 模式：连接第三方日历，而密钥从不接触客户端
服务器需要获得某个第三方日历的 OAuth 授权。它发送一个 <strong>URL 模式</strong>征询（URL-mode Elicitation），并把一个流程 id 保存在 `requestState` 中。客户端的回调显示完整的 URL 和域名，征得同意后“打开浏览器”。我们的模拟浏览器通过直接调用服务器的回调端点来完成这个流程。客户端始终只返回 `{"action": "accept"}`。重试时，服务器在<strong>自己的令牌存储</strong>中查找这个流程（正如规范对第三方令牌所说的，这部分是有状态的），然后要么完成，要么再次询问。

In [ ]:
import uuid
from urllib.parse import parse_qs, urlparse

calendar = MCPServer("calendar")
PENDING_FLOWS: dict[str, str] = {}      # flow_id -> user (server-side, bound to the authenticated user)
TOKENS: dict[str, str] = {}             # user -> third-party token (never sent to the client)


def oauth_callback(flow_id: str):
    """What the server's https://.../connect route would do after the third-party redirect."""
    user = PENDING_FLOWS.pop(flow_id)
    TOKENS[user] = f"tp-token-{secrets.token_hex(4)}"


@calendar.tool()
def next_meeting(ctx: Context) -> str | types.InputRequiredResult:
    """Return the user's next meeting from their third-party calendar."""
    user = "user-123"                                           # in production: the `sub` from MCP authorization
    if user in TOKENS:
        return f"Next meeting: Design review at 15:00 (fetched with a token the client never saw)"
    if ctx.request_state:                                       # retried, but the browser flow isn't done yet
        flow_id = json.loads(ctx.request_state)["flow"]
    else:
        flow_id = uuid.uuid4().hex[:8]
        PENDING_FLOWS[flow_id] = user
    return types.InputRequiredResult(
        input_requests={"connect": types.ElicitRequest(params=types.ElicitRequestURLParams(
            message="Connect your calendar so I can read your meetings.",
            url=f"https://mcp.example.com/connect?flow={flow_id}"))},
        request_state=json.dumps({"flow": flow_id}))


async def url_consent(context, params):
    if params.mode != "url":
        return types.ElicitResult(action="decline")
    domain = urlparse(params.url).hostname
    print(f"   [consent] calendar server wants to open: {params.url}\n   [consent] domain: {domain} -> user approves")
    flow = parse_qs(urlparse(params.url).query)["flow"][0]
    oauth_callback(flow)                                         # stands in for the user finishing OAuth in a browser
    return types.ElicitResult(action="accept")                   # no content in URL mode

async with Client(calendar, elicitation_callback=url_consent) as client:
    first = await client.session.call_tool("next_meeting", {}, allow_input_required=True)
    print("input request:", wire(first.input_requests["connect"]))
    print("\nauto loop ->", (await client.call_tool("next_meeting", {})).content[0].text)
    print("client-visible data contained a token?", any("tp-token" in json.dumps(v) for v in [wire(first)]))

这里没有 `elicitationId`，也没有 `notifications/elicitation/complete`。重试本身<em>就是</em>完成信号。

### 8. 声明式的做法：由解析器负责簿记
对于大多数工具，你并不需要手工管理状态。一个 `Resolve(...)` 解析器（Resolver）返回 `Elicit(message, Model)`。SDK 会构建 `InputRequiredResult`，把答案记录在 `requestState` 中以确保每个问题只问一次，检查客户端的能力（缺失时返回 `-32021`），并且只在所有问题都解决后才运行函数体。这里的两个解析器意味着<strong>一轮中有两个输入请求</strong>。

In [ ]:
from typing import Annotated

from pydantic import BaseModel, Field

from mcp.server.elicitation import AcceptedElicitation, ElicitationResult
from mcp.server.mcpserver import Elicit, Resolve


class Environment(BaseModel):
    env: str = Field(json_schema_extra={"enum": ["staging", "production"]}, description="Target environment")


class Approval(BaseModel):
    approve: bool = Field(description="Approve this deployment?")


deployer = MCPServer("deployer")


def pick_env() -> Elicit[Environment]:
    return Elicit("Which environment?", Environment)


def approve(service: str) -> Elicit[Approval]:
    return Elicit(f"Deploy {service}? This restarts it.", Approval)


@deployer.tool()
def deploy(service: str,
           target: Annotated[Environment, Resolve(pick_env)],
           ok: Annotated[ElicitationResult[Approval], Resolve(approve)]) -> str:
    """Deploy a service after asking for the environment and an approval."""
    if isinstance(ok, AcceptedElicitation) and ok.data.approve:
        return f"Deployed {service} to {target.env}."
    return f"Deployment of {service} not approved ({ok.action})."


async def deploy_ui(context, params):
    props = params.requested_schema["properties"]
    content = {"env": "staging"} if "env" in props else {"approve": True}
    print(f"   [dialog] {params.message} -> {content}")
    return types.ElicitResult(action="accept", content=content)

async with Client(deployer, elicitation_callback=deploy_ui) as client:
    first = await client.session.call_tool("deploy", {"service": "api"}, allow_input_required=True)
    print("one round, input request keys:", list(first.input_requests))
    print((await client.call_tool("deploy", {"service": "api"})).content[0].text)

async with Client(deployer) as no_ui:
    try:
        await no_ui.call_tool("deploy", {"service": "api"})
    except MCPError as err:
        print("client without elicitation ->", err.error.code, err.error.message[:70])

### 动手练习
1. 自己编写一个客户端的 MRTR 驱动程序：围绕 `client.session.call_tool(..., allow_input_required=True)` 写一个 `while isinstance(result, types.InputRequiredResult)` 循环，把每个 `inputRequests` 条目分派给你自己的函数，最多 5 轮，并在没有 `inputRequests` 时进行退避。用 `delete_repository` 的输错场景来测试它。
2. 让第 1 节的 `delete_repository` 做到<strong>一次性使用安全</strong>：把它完成过的每个 `requestState` 记录到一个 `USED` 集合中（以令牌的哈希作为键），并拒绝第二次完成。证明即使在 TTL 之内，重放第 2 轮也会被拒绝。
3. 在一个手工编写的请求中，使用 `oneOf: [{"const": "stg", "title": "Staging"}, ...]`，把第 8 节中的 `Environment` 模式改成带标题的单选枚举，并在你的回调中用这些标题来渲染它。
4. 设计评审：一位同事的服务器“因为更快”而通过表单模式征询索取用户的 AWS 秘密密钥。列出它违反了哪些规范规则，并用 URL 模式重新设计它，包括你将如何防御网络钓鱼攻击。